# P2 RAG Chunk Quick Check

`parsing_p2_250` 산출물의 chunk_id, toc 분리, artifact, smoke search를 빠르게 점검합니다.


In [1]:
from pathlib import Path
import json
import re
import collections

import pandas as pd

OUTPUT_DIR = Path.cwd()
if not (OUTPUT_DIR / "chunks_v2.jsonl").exists():
    OUTPUT_DIR = Path(r"C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250")

SUMMARY_PATH = OUTPUT_DIR / "parsing_summary.json"
METADATA_EXCEL_PATH = OUTPUT_DIR / "rfp_parsing_metadata_250_p2_chunkfix_toc_clean.xlsx"
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SUMMARY_PATH:", SUMMARY_PATH)
print("METADATA_EXCEL_PATH:", METADATA_EXCEL_PATH)


OUTPUT_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250
SUMMARY_PATH: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\parsing_summary.json
METADATA_EXCEL_PATH: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_p2_250\rfp_parsing_metadata_250_p2_chunkfix_toc_clean.xlsx


In [2]:
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
metadata_df = pd.read_excel(METADATA_EXCEL_PATH, sheet_name="metadata_250")
print("output_description:", summary.get("output_description"))
print("parsing_output_name:", summary.get("parsing_output_name"))
print("parsing_version_label:", summary.get("parsing_version_label"))
print("metadata rows:", len(metadata_df))
print("parse_success_docs:", summary.get("parse_success_docs"))
print("v1 chunks:", summary.get("v1_chunks"))
print("v2 chunks:", summary.get("v2_chunks"))


output_description: P2 - chunk_id 중복 수정, toc 분리, artifact cleaner 보강
parsing_output_name: parsing_p2_250
parsing_version_label: p2_chunkfix_toc_clean
metadata rows: 250
parse_success_docs: 250
v1 chunks: 31641
v2 chunks: 41885


In [3]:
WEIRD_PATTERN = re.compile(
    r"[\u0100-\u017F][\u3000-\u9FFF]"
    r"|[\u3000-\u9FFF][\u0100-\u017F]"
    r"|[\u0F00-\u0FFF]"
    r"|[\u0C00-\u0C7F]"
    r"|[\u0800-\u08FF]"
)
REQUIRED_COLUMNS = [
    "chunk_id", "source_file", "project_name", "issuer", "chunk_type",
    "section_type", "content", "embed_enabled"
]
KEEP_HANJA = ''.join(chr(x) for x in [0x73FE, 0x7121, 0x6709, 0x4E59, 0x65B0, 0x820A, 0x5167, 0x5916])

def scan_chunks(path: Path) -> dict:
    ids = set()
    duplicate_count = 0
    missing = collections.Counter()
    chunk_types = collections.Counter()
    embed_enabled = collections.Counter()
    weird_count = 0
    docs = set()
    keep_hanja_hits = collections.Counter()
    rows = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rows += 1
            row = json.loads(line)
            chunk_id = row.get("chunk_id")
            if chunk_id in ids:
                duplicate_count += 1
            ids.add(chunk_id)
            docs.add(row.get("source_file", ""))
            for col in REQUIRED_COLUMNS:
                if col != "embed_enabled" and not str(row.get(col, "")).strip():
                    missing[col] += 1
            chunk_types[row.get("chunk_type", "")] += 1
            embed_enabled[str(row.get("embed_enabled"))] += 1
            content = str(row.get("content", ""))
            if WEIRD_PATTERN.search(content):
                weird_count += 1
            for ch in KEEP_HANJA:
                if ch in content:
                    keep_hanja_hits[f"U+{ord(ch):04X}"] += 1
    return {
        "rows": rows,
        "unique_ids": len(ids),
        "unique_docs": len(docs),
        "duplicate_chunk_id": duplicate_count,
        "chunk_type_counts": dict(chunk_types),
        "embed_enabled_counts": dict(embed_enabled),
        "missing_required": dict(missing),
        "weird_unicode_chunks": weird_count,
        "normal_hanja_hits": dict(keep_hanja_hits),
    }

for filename in ["chunks_v1.jsonl", "chunks_v2.jsonl"]:
    print("\n===", filename, "===")
    result = scan_chunks(OUTPUT_DIR / filename)
    for key, value in result.items():
        print(f"{key}: {value}")



=== chunks_v1.jsonl ===


rows: 31641
unique_ids: 31641
unique_docs: 250
duplicate_chunk_id: 0
chunk_type_counts: {'toc': 248, 'text': 31393}
embed_enabled_counts: {'False': 248, 'True': 31393}
missing_required: {}
weird_unicode_chunks: 0
normal_hanja_hits: {'U+65B0': 16, 'U+5167': 53, 'U+5916': 11, 'U+73FE': 28, 'U+820A': 69, 'U+4E59': 1, 'U+7121': 2}

=== chunks_v2.jsonl ===


rows: 41885
unique_ids: 41885
unique_docs: 250
duplicate_chunk_id: 0
chunk_type_counts: {'toc': 248, 'text': 31393, 'table': 7369, 'fact_candidates': 2875}
embed_enabled_counts: {'False': 248, 'True': 41637}
missing_required: {}
weird_unicode_chunks: 0
normal_hanja_hits: {'U+65B0': 18, 'U+5167': 55, 'U+5916': 11, 'U+73FE': 33, 'U+820A': 102, 'U+4E59': 1, 'U+7121': 2}


In [4]:
queries = [
    "\u0032\u0038\uac1c\uc6d4",
    "\uacc4\uc57d \uccb4\uacb0\uc77c\ub85c\ubd80\ud130 \u0032\u0038\uac1c\uc6d4",
    "\uacc4\uc57d\uc77c\ub85c\ubd80\ud130 \u0032\u0038\uac1c\uc6d4",
    "\uc785\ucc30\ucc38\uac00\uc790\uaca9",
    "\uc81c\ucd9c\uc11c\ub958",
    "195,030,000",
]

for query in queries:
    hits = []
    with (OUTPUT_DIR / "chunks_v2.jsonl").open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if query in str(row.get("content", "")):
                hits.append({
                    "chunk_id": row.get("chunk_id"),
                    "source_file": row.get("source_file"),
                    "chunk_type": row.get("chunk_type"),
                    "section_type": row.get("section_type"),
                })
                if len(hits) >= 5:
                    break
    print()
    print("QUERY:", query, "| hits:", len(hits))
    display(pd.DataFrame(hits))



QUERY: 28개월 | hits: 5


,chunk_id,source_file,chunk_type,section_type
0,P009_v2_text_0004_part_001,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,text,사업개요
1,P009_v2_text_0004_part_002,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,text,사업개요
2,P009_v2_text_0004_part_003,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,text,사업개요
3,P009_v2_fact_0001_part_001,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,fact_candidates,추출필드후보
4,P009_v2_fact_0001_part_002,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,fact_candidates,추출필드후보



QUERY: 계약 체결일로부터 28개월 | hits: 4


,chunk_id,source_file,chunk_type,section_type
0,P009_v2_text_0004_part_002,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,text,사업개요
1,P009_v2_text_0004_part_003,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,text,사업개요
2,P009_v2_fact_0001_part_001,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,fact_candidates,추출필드후보
3,P009_v2_fact_0001_part_002,한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp,fact_candidates,추출필드후보



QUERY: 계약일로부터 28개월 | hits: 3


...

,chunk_id,source_file,chunk_type,section_type
0,P001_v2_text_0084_part_002,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,입찰참가자격
1,P001_v2_fact_0001_part_004,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,fact_candidates,추출필드후보
2,P001_v2_fact_0001_part_005,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,fact_candidates,추출필드후보
3,P001_v2_fact_0001_part_006,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,fact_candidates,추출필드후보
4,P002_v2_text_0004_part_001,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,text,입찰참가자격



QUERY: 제출서류 | hits: 5


,chunk_id,source_file,chunk_type,section_type
0,P001_v2_text_0029_part_003,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,제안요청내용
1,P001_v2_text_0030_part_001,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,제안요청내용
2,P001_v2_text_0030_part_003,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,제안요청내용
3,P001_v2_text_0032_part_002,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,제안요청내용
4,P001_v2_text_0043_part_001,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,text,제안요청내용



QUERY: 195,030,000 | hits: 3


,chunk_id,source_file,chunk_type,section_type
0,P013_v2_text_0002_part_001,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp,text,사업개요
1,P013_v2_fact_0001_part_001,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp,fact_candidates,추출필드후보
2,P013_v2_fact_0001_part_002,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp,fact_candidates,추출필드후보
